In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [2]:
with open("dataset/dataset.pkl", "rb") as f:
    dataset = pickle.load(f)
    x_files = dataset["gtr"]
    y_files = dataset["ney"]

In [3]:
x_train_files, x_test_files, y_train_files, y_test_files = train_test_split(
    x_files, y_files, test_size=0.2, random_state=42)

print(len(x_train_files), len(x_test_files))

200 50


In [4]:
print(x_train_files[0], y_train_files[0])
print(x_test_files[0], y_test_files[0])

gtr_132.npy ney_132.npy
gtr_142.npy ney_142.npy


In [5]:
NEY_SPECTROGRAM_DIR = "dataset/spectrograms/ney/"
GTR_SPECTROGRAM_DIR = "dataset/spectrograms/ac_gtr/"

In [6]:
class SpectrogramDataset(Dataset):
    def __init__(self,
                 x_files,
                 y_files,
                 ney_spectrogram_dir,
                 gtr_spectrogram_dir):
        self.x_files = x_files
        self.y_files = y_files
        self.ney_spectrogram_dir = ney_spectrogram_dir
        self.gtr_spectrogram_dir = gtr_spectrogram_dir
        self.x = []
        self.y = []

        # gtr files
        for f in self.x_files:
            spectrogram = np.load(
                self.gtr_spectrogram_dir + f, allow_pickle=True)
            self.x.append(spectrogram)

        # ney files
        for f in self.y_files:
            spectrogram = np.load(
                self.ney_spectrogram_dir + f, allow_pickle=True)
            self.y.append(spectrogram)

        # prepare gtr spectrograms
        self.x = np.array(self.x)
        # min max scaling
        self.x = (self.x - np.min(self.x)) / (np.max(self.x) - np.min(self.x))
        # expand dims to imitate grayscale img format
        self.x = np.expand_dims(self.x, axis=3)
        
        # prepare ney spectrograms
        self.y = np.array(self.y)
        # min max scaling
        self.y = (self.y - np.min(self.y)) / (np.max(self.y) - np.min(self.y))
        # expand dims to imitate grayscale img format
        self.y = np.expand_dims(self.y, axis=3)

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return self.x.shape[0]

In [7]:
train_dataset = SpectrogramDataset(x_files,
                                   y_files,
                                   NEY_SPECTROGRAM_DIR,
                                   GTR_SPECTROGRAM_DIR)

In [8]:
train_data_loader = DataLoader(
    dataset=train_dataset,
    batch_size=16,
    shuffle=True)

In [9]:
x, y = next(iter(train_data_loader))

In [10]:
x.shape

torch.Size([16, 256, 40, 1])